# 02 — Cleaning del dataset LAPD

**Fase PACE**: Analyze  
**Obiettivo**: applicare la pulizia completa al dataset combinato e produrre 
`crimes_clean.parquet`, base di riferimento per tutti i notebook successivi.

**Input**: `data/processed/crimes_merged.parquet` (prodotto da `01_data_loading.ipynb`)  
**Output**: `data/processed/crimes_clean.parquet`

## 1. Setup e caricamento

In [1]:
import pandas as pd  # Importazione framework Pandas per la manipolazione del DataFrame
import numpy as np   # Importazione framework Numpy per gestire le operazioni con i numeri

pd.set_option('display.max_rows', None)         # Impostazione pandas per mostrare tutte le righe con i comandi di ispezione
pd.set_option('display.max_columns', None)      # Impostazione pandas per mostrare tutte le colonne con i comandi di ispezione
pd.set_option('display.max_info_columns', 200)  # Impostazione pandas per mostrare le informazioni di 200 colonne con il comando '.info()'

df = pd.read_parquet('../../data/processed/crimes_merged.parquet') # Creazione della variabile 'df' che conterrà i dati presenti nel
                                                                   # file parquet 'crimes_merged.parquet'

print(f"Dataset caricato: {df.shape}")                             # Visualizza il numero di righe e colonne del DataFrame

print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")  # Visualizza la quantità di memoria utilizzata per eseguire 
                                                                        # le operazioni precedenti

Dataset caricato: (3138031, 28)
Memoria: 2659.9 MB


## 2. Eliminazione colonne non rilevanti

In base all'ispezione iniziale (fase Plan) eliminiamo le colonne che non 
porteranno valore alle analisi:

- **`Crm Cd 2`, `Crm Cd 3`, `Crm Cd 4`** (>93% nulli): rappresentano crimini 
  secondari collegati al report principale, ma sono praticamente sempre vuote. 
  Per le nostre domande analitiche è sufficiente `Crm Cd` (crimine principale).
- **`Cross Street`** (84% nulli): informazione ridondante rispetto a `LOCATION` 
  e alle coordinate `LAT`/`LON`.

In [2]:
colonne_da_eliminare = ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']

df = df.drop(columns=colonne_da_eliminare)

print(f"Colonne eliminate: {colonne_da_eliminare}")
print(f"Nuova shape: {df.shape}")
print(f"Colonne rimanenti: {df.shape[1]}")

Colonne eliminate: ['Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'Cross Street']
Nuova shape: (3138031, 24)
Colonne rimanenti: 24


## 3. Conversione dei tipi di dato

### 3.1 Date (`Date Rptd` e `DATE OCC`)

Entrambe le colonne sono attualmente stringhe nel formato `MM/DD/YYYY HH:MM:SS AM/PM` 
(formato US). Le convertiamo in `datetime64` per poter eseguire operazioni temporali, 
filtri per anno/mese, calcoli di intervalli, ecc.

- `DATE OCC` = data in cui è avvenuto il crimine (la più importante per le analisi)
- `Date Rptd` = data in cui il crimine è stato denunciato/registrato (può essere 
  successiva a quella di occorrenza, e il delta è un'informazione interessante)

In [3]:
df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')   # Conversione della colonna 'DATE OCC' nel formato datetime
df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce') # Conversione della colonna 'Date Rptd' nel formato datetime

print("Tipi dopo la conversione:")  # Visualizza il testo indicato

print(df[['DATE OCC', 'Date Rptd']].dtypes)  # Visualizza il dtype delle colonne 'DATE OCC' e 'Date Rptd'

print(f"\nNulli DATE OCC: {df['DATE OCC'].isnull().sum()}")  # Mostra il numero di valori nulli della colonna 'DATE OCC'

print(f"Nulli Date Rptd: {df['Date Rptd'].isnull().sum()}")  # Mostra il numero di valori nulli della colonna 'Date Rptd'

print(f"\nRange DATE OCC: {df['DATE OCC'].min()} → {df['DATE OCC'].max()}")  # Visualizza i valori minimo e massimo della colonna 'DATE OCC'
print(f"Range Date Rptd: {df['Date Rptd'].min()} → {df['Date Rptd'].max()}") # Visualizza i valori minimo e massimo della colonna 'Date Rptd'

/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_2812/2853255494.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['DATE OCC'] = pd.to_datetime(df['DATE OCC'], errors='coerce')   # Conversione della colonna 'DATE OCC' nel formato datetime
/var/folders/wl/_8d4j_cs57qgchs9ckhqzkyr0000gn/T/ipykernel_2812/2853255494.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date Rptd'] = pd.to_datetime(df['Date Rptd'], errors='coerce') # Conversione della colonna 'Date Rptd' nel formato datetime


Tipi dopo la conversione:
DATE OCC     datetime64[ns]
Date Rptd    datetime64[ns]
dtype: object

Nulli DATE OCC: 0
Nulli Date Rptd: 0

Range DATE OCC: 2010-01-01 00:00:00 → 2024-12-30 00:00:00
Range Date Rptd: 2010-01-01 00:00:00 → 2025-06-05 00:00:00


### 3.2 Orario (`TIME OCC`)

`TIME OCC` è attualmente un intero nel formato HHMM (es. `2130` = 21:30, 
`45` = 00:45, `0` = 00:00). Lo trasformiamo in qualcosa di più gestibile: 
una colonna `hour_occ` con solo l'ora intera (0-23), che è la granularità 
sufficiente per analisi tipo "fasce orarie più critiche" o "distribuzione 
oraria dei crimini".

La colonna originale `TIME OCC` viene sostituita.

In [4]:
# Estrazione dell'ora dall'intero HHMM
df['hour_occ'] = df['TIME OCC'] // 100 # Applicazione dell'operatore 'divisione intera' alla colonna 'TIME OCC' per estrarre i valori
                                       # che comporranno la colonna 'hour_occ'

# Verifica
print(f"Range hour_occ: {df['hour_occ'].min()} → {df['hour_occ'].max()}") # Operazione di controllo che visualizza i numeri minimo e massimo
                                                                          # della colonna 'hour_occ'

print(f"\nDistribuzione per ora (top 5):")                                # Visualizza la stringa indicata

print(df['hour_occ'].value_counts().sort_index().head(5))                 # Visualizza le prime 5 righe della colonna 'hour_occ'
                                                                          # ordinati in modo ascendente in base all'indice

print(f"\nValori fuori range (>23 o <0): {((df['hour_occ'] < 0) | (df['hour_occ'] > 23)).sum()}") 
# Visualizza l f-string indicata che contiene del testo e la somma dei valori della colonna 'hour_occ' minori di 0 e maggiori di 23
# per controllare che tutti i valori siano compresi nell'intervallo del massimo e minimo creato in precedenza 
                                                                                                    

Range hour_occ: 0 → 23

Distribuzione per ora (top 5):
hour_occ
0    130102
1     90502
2     76769
3     60937
4     48319
Name: count, dtype: int64

Valori fuori range (>23 o <0): 0


In [5]:
# Elimina la colonna 'TIME OCC' dal DataFrame e ricalcola la dimensione del DataFrame tramite '.shape'
df = df.drop(columns=['TIME OCC']) 
print(f"Colonna 'TIME OCC' eliminata. Shape attuale: {df.shape}")

Colonna 'TIME OCC' eliminata. Shape attuale: (3138031, 24)


### 3.3 Gestione coordinate sentinella

Nel notebook 01 abbiamo identificato 3.148 record con coordinate `(0, 0)`, 
che il LAPD usa come valore sentinella per indicare "posizione sconosciuta".

Le sostituiamo con `NaN` invece di eliminare i record: in questo modo i record 
restano disponibili per analisi non-geografiche (tipologia di crimine, vittime, 
orari) ma vengono automaticamente esclusi dalle analisi che richiedono 
le coordinate.

In [6]:
maschera_sentinella = (df['LAT'] == 0) & (df['LON'] == 0)
n_sentinelle = maschera_sentinella.sum()

df.loc[maschera_sentinella, ['LAT', 'LON']] = np.nan

print(f"Record con coordinate sentinella sostituite: {n_sentinelle}")
print(f"Nulli LAT dopo sostituzione: {df['LAT'].isnull().sum()}")
print(f"Nulli LON dopo sostituzione: {df['LON'].isnull().sum()}")
print(f"\nNuovo range LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"Nuovo range LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")

Record con coordinate sentinella sostituite: 3148
Nulli LAT dopo sostituzione: 3148
Nulli LON dopo sostituzione: 3148

Nuovo range LAT: 33.3427 → 34.7907
Nuovo range LON: -118.8279 → -117.6596


## 4. Gestione duplicati su DR_NO

`DR_NO` (Division of Records Number) è l'identificativo univoco del report 
di polizia. In teoria non dovrebbero esserci duplicati, ma nell'ispezione 
iniziale ne abbiamo trovati 57.809.

Prima di decidere come gestirli, ispezioniamo alcuni casi reali per capire 
**perché** esistono e in cosa differiscono le righe duplicate.

In [7]:
# Trova un esempio di DR_NO duplicato
dr_no_duplicati = df[df.duplicated(subset='DR_NO', keep=False)]['DR_NO'].unique()
print(f"DR_NO unici duplicati: {len(dr_no_duplicati)}")
print(f"Righe totali coinvolte: {df.duplicated(subset='DR_NO', keep=False).sum()}")

# Prendi i primi 3 DR_NO duplicati e mostrali
print("\n=== Primi 3 esempi di duplicati ===\n")
for dr_no in dr_no_duplicati[:3]:
    print(f"--- DR_NO: {dr_no} ---")
    print(df[df['DR_NO'] == dr_no])
    print()

DR_NO unici duplicati: 57809
Righe totali coinvolte: 115618

=== Primi 3 esempi di duplicati ===

--- DR_NO: 161804259 ---
             DR_NO  Date Rptd   DATE OCC  AREA  AREA NAME  Rpt Dist No  \
937560   161804259 2016-01-06 2016-01-06    18  Southeast         1805   
1234823  161804259 2016-01-06 2016-01-06    18  Southeast         1805   

         Part 1-2  Crm Cd       Crm Cd Desc Mocodes  Vict Age Vict Sex  \
937560          1     510  VEHICLE - STOLEN    None         0     None   
1234823         1     510  VEHICLE - STOLEN    None         0     None   

        Vict Descent  Premis Cd Premis Desc  Weapon Used Cd Weapon Desc  \
937560          None      101.0      STREET             NaN        None   
1234823         None      101.0      STREET             NaN        None   

        Status  Status Desc  Crm Cd 1                                LOCATION  \
937560      IC  Invest Cont     510.0  200 E  90TH                         ST   
1234823     IC  Invest Cont     510.0  200 

In [8]:
# Verifica se i duplicati su DR_NO sono anche duplicati esatti su tutta la riga
duplicati_su_dr_no = df.duplicated(subset='DR_NO', keep=False).sum()
duplicati_esatti_riga = df.duplicated(keep=False).sum()

print(f"Righe con DR_NO duplicato: {duplicati_su_dr_no}")
print(f"Righe duplicate esatte (tutte le colonne): {duplicati_esatti_riga}")

if duplicati_su_dr_no == duplicati_esatti_riga:
    print("\n✅ Tutti i duplicati su DR_NO sono duplicati esatti")
else:
    diff = duplicati_su_dr_no - duplicati_esatti_riga
    print(f"\n⚠️ Ci sono {diff} righe con DR_NO duplicato ma con differenze nelle colonne")

Righe con DR_NO duplicato: 115618
Righe duplicate esatte (tutte le colonne): 115618

✅ Tutti i duplicati su DR_NO sono duplicati esatti


In [9]:
duplicati_df = df[df.duplicated(subset='DR_NO', keep=False)]
print("Top 10 tipi di crimine tra i duplicati:")
print(duplicati_df['Crm Cd Desc'].value_counts().head(10))
print(f"\nTotale tipi di crimine distinti tra i duplicati: {duplicati_df['Crm Cd Desc'].nunique()}")

Top 10 tipi di crimine tra i duplicati:
Crm Cd Desc
THEFT OF IDENTITY                                          10608
VEHICLE - STOLEN                                            8792
BATTERY - SIMPLE ASSAULT                                    8086
BURGLARY FROM VEHICLE                                       7730
BURGLARY                                                    7698
THEFT PLAIN - PETTY ($950 & UNDER)                          6864
INTIMATE PARTNER - SIMPLE ASSAULT                           6464
VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS)     5974
THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER)             5348
ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT              4726
Name: count, dtype: int64

Totale tipi di crimine distinti tra i duplicati: 122


### Rimozione dei duplicati

Tutti i 57.809 DR_NO duplicati risultano essere duplicati esatti (righe 
identiche su tutte le colonne), distribuiti su 122 tipi di crimine diversi. 
Si tratta quindi di un errore generalizzato di caricamento nel dataset 
originale, non di un problema specifico per categoria.

Rimuoviamo i duplicati mantenendo la prima occorrenza di ogni riga.

In [10]:
prima = len(df)
df = df.drop_duplicates()
dopo = len(df)

print(f"Righe prima: {prima}")
print(f"Righe dopo: {dopo}")
print(f"Righe rimosse: {prima - dopo}")
print(f"\nDuplicati residui su DR_NO: {df['DR_NO'].duplicated().sum()}")

Righe prima: 3138031
Righe dopo: 3080222
Righe rimosse: 57809

Duplicati residui su DR_NO: 0


## 5. Gestione valori nulli

Aggiorniamo il quadro dei nulli dopo l'eliminazione delle colonne e dei duplicati.

In [11]:
nulli_pct = (df.isnull().sum() / len(df) * 100).round(2)
nulli_pct = nulli_pct[nulli_pct > 0].sort_values(ascending=False)
print("Colonne con valori nulli:")
print(nulli_pct)

Colonne con valori nulli:
Weapon Used Cd    66.73
Weapon Desc       66.73
Mocodes           12.18
Vict Sex          10.95
Vict Descent      10.95
LAT                0.10
LON                0.10
Premis Desc        0.03
dtype: float64


### 5.1 Weapon Used Cd / Weapon Desc

Il 66.73% di nulli non indica dati mancanti: la maggior parte dei crimini 
semplicemente non prevede un'arma (furti, frodi, vandalismi, ecc.). 
Ricodifichiamo i NaN come "No Weapon" nella descrizione e come `0` nel codice.

In [12]:
df['Weapon Desc'] = df['Weapon Desc'].fillna('No Weapon')
df['Weapon Used Cd'] = df['Weapon Used Cd'].fillna(0)

print("Weapon Desc — valori più frequenti:")
print(df['Weapon Desc'].value_counts().head(5))
print(f"\nNulli Weapon Desc: {df['Weapon Desc'].isnull().sum()}")
print(f"Nulli Weapon Used Cd: {df['Weapon Used Cd'].isnull().sum()}")

Weapon Desc — valori più frequenti:
Weapon Desc
No Weapon                                         2055380
STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)     598043
UNKNOWN WEAPON/OTHER WEAPON                         92736
VERBAL THREAT                                       81418
HAND GUN                                            53593
Name: count, dtype: int64

Nulli Weapon Desc: 0
Nulli Weapon Used Cd: 0


### 5.2 Vict Sex / Vict Descent

Il ~11% di nulli rappresenta casi in cui la vittima non è stata identificata 
o il crimine non ha una vittima diretta (es. furto di veicolo, vandalismo). 
Ricodifichiamo come "X" (Unknown) per entrambe le colonne, coerente con 
la codifica già usata dal LAPD per i casi incerti.

df['Vict Sex'] = df['Vict Sex'].fillna('X')
df['Vict Descent'] = df['Vict Descent'].fillna('X')

print("Vict Sex — distribuzione:")
print(df['Vict Sex'].value_counts())
print(f"\nVict Descent — valori unici: {df['Vict Descent'].nunique()}")
print(f"Nulli Vict Sex: {df['Vict Sex'].isnull().sum()}")
print(f"Nulli Vict Descent: {df['Vict Descent'].isnull().sum()}")

### 5.3 Mocodes

I Mocodes (codici di modus operandi) sono nulli nel 12% dei casi. 
Questo è un valore mancante legittimo: non tutti i crimini hanno un modus 
operandi codificato. Li lasciamo come NaN e li gestiremo specificamente 
nel notebook di analisi dedicato ai pattern dei modus operandi.

In [14]:
print(f"Mocodes — nulli: {df['Mocodes'].isnull().sum()} ({df['Mocodes'].isnull().mean()*100:.2f}%)")
print(f"Esempio di Mocodes non nullo: {df['Mocodes'].dropna().iloc[0]}")

Mocodes — nulli: 375261 (12.18%)
Esempio di Mocodes non nullo: 0913 1814 2000


### 5.4 Premis Desc

Solo lo 0.03% di nulli. Percentuale trascurabile. Rimuoviamo queste poche 
righe perché non vale la pena imputare un valore e la perdita è irrilevante.

In [15]:
prima = len(df)
df = df.dropna(subset=['Premis Desc'])
print(f"Righe rimosse per Premis Desc nullo: {prima - len(df)}")
print(f"Shape attuale: {df.shape}")

Righe rimosse per Premis Desc nullo: 775
Shape attuale: (3079447, 24)


### 5.5 Pulizia valori anomali in Vict Sex

Oltre a M, F, X sono presenti valori rari: `H` (185), `N` (17), `-` (2). 
Ricodifichiamo tutti come `X` (Unknown) per semplicità, dato che rappresentano 
lo 0.007% del dataset e non sono sufficienti per analisi statistiche 
significative sulla categoria.

In [16]:
valori_validi = ['M', 'F', 'X']
df.loc[~df['Vict Sex'].isin(valori_validi), 'Vict Sex'] = 'X'

print("Vict Sex dopo pulizia:")
print(df['Vict Sex'].value_counts())

Vict Sex dopo pulizia:
Vict Sex
M    1359854
F    1230571
X     489022
Name: count, dtype: int64


## 6. Gestione valori sentinella in Vict Age

`Vict Age` è un intero che dovrebbe rappresentare l'età della vittima. 
Tuttavia il LAPD usa il valore `0` per indicare "età sconosciuta" (stessa 
logica delle coordinate sentinella). Potrebbero esserci anche valori negativi 
o implausibilmente alti. Ispezioniamo prima di decidere.

In [17]:
print("Statistiche Vict Age:")
print(df['Vict Age'].describe())
print(f"\nValori <= 0: {(df['Vict Age'] <= 0).sum()}")
print(f"Valori > 120: {(df['Vict Age'] > 120).sum()}")
print(f"\nDistribuzione valori anomali:")
print(df[df['Vict Age'] <= 0]['Vict Age'].value_counts().sort_index())

Statistiche Vict Age:
count    3.079447e+06
mean     3.081706e+01
std      2.113740e+01
min     -1.300000e+01
25%      1.800000e+01
50%      3.100000e+01
75%      4.600000e+01
max      1.200000e+02
Name: Vict Age, dtype: float64

Valori <= 0: 632399
Valori > 120: 0

Distribuzione valori anomali:
Vict Age
-13         1
-12         3
-11         2
-10        12
-9         18
-8         13
-7         18
-6         25
-5         38
-4         51
-3         79
-2        152
-1        366
 0     631621
Name: count, dtype: int64


### 6.1 Ricodifica valori sentinella e anomali

- `Vict Age = 0` (631.621 record, ~20.5%): valore sentinella per "età sconosciuta". 
  Coerente con crimini senza vittima diretta.
- `Vict Age < 0` (778 record): errori di inserimento dati.

Sostituiamo tutti i valori ≤ 0 con `NaN`: i record restano disponibili per 
analisi non-demografiche, ma vengono esclusi automaticamente dai calcoli 
su età (media, mediana, distribuzione, ecc.).

In [18]:
maschera = df['Vict Age'] <= 0
n_sostituiti = maschera.sum()

df.loc[maschera, 'Vict Age'] = np.nan

print(f"Valori sostituiti con NaN: {n_sostituiti}")
print(f"\nStatistiche Vict Age dopo pulizia:")
print(df['Vict Age'].describe())
print(f"\nNulli Vict Age: {df['Vict Age'].isnull().sum()}")

Valori sostituiti con NaN: 632399

Statistiche Vict Age dopo pulizia:
count    2.447048e+06
mean     3.878205e+01
std      1.591625e+01
min      2.000000e+00
25%      2.700000e+01
50%      3.600000e+01
75%      5.000000e+01
max      1.200000e+02
Name: Vict Age, dtype: float64

Nulli Vict Age: 632399


## 7. Verifica finale

Riepilogo dello stato del dataset dopo tutte le operazioni di cleaning.

In [19]:
print("=" * 50)
print("VERIFICA FINALE — DATASET PULITO")
print("=" * 50)

print(f"\nShape: {df.shape}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Duplicati su DR_NO: {df['DR_NO'].duplicated().sum()}")

print(f"\n--- Tipi di dato ---")
print(df.dtypes)

print(f"\n--- Valori nulli ---")
nulli = df.isnull().sum()
nulli = nulli[nulli > 0]
for col, n in nulli.items():
    print(f"  {col}: {n} ({n/len(df)*100:.2f}%)")

if len(nulli) == 0:
    print("  Nessun valore nullo")

print(f"\n--- Range temporale ---")
print(f"  DATE OCC: {df['DATE OCC'].min().date()} → {df['DATE OCC'].max().date()}")
print(f"  Date Rptd: {df['Date Rptd'].min().date()} → {df['Date Rptd'].max().date()}")

print(f"\n--- Range geografico ---")
print(f"  LAT: {df['LAT'].min():.4f} → {df['LAT'].max():.4f}")
print(f"  LON: {df['LON'].min():.4f} → {df['LON'].max():.4f}")

print(f"\n--- Conteggi chiave ---")
print(f"  Anni coperti: {df['DATE OCC'].dt.year.nunique()}")
print(f"  Aree LAPD: {df['AREA NAME'].nunique()}")
print(f"  Tipi di crimine: {df['Crm Cd Desc'].nunique()}")

VERIFICA FINALE — DATASET PULITO

Shape: (3079447, 24)
Memoria: 2180.7 MB
Duplicati su DR_NO: 0

--- Tipi di dato ---
DR_NO                      int64
Date Rptd         datetime64[ns]
DATE OCC          datetime64[ns]
AREA                       int64
AREA NAME                 object
Rpt Dist No                int64
Part 1-2                   int64
Crm Cd                     int64
Crm Cd Desc               object
Mocodes                   object
Vict Age                 float64
Vict Sex                  object
Vict Descent              object
Premis Cd                float64
Premis Desc               object
Weapon Used Cd           float64
Weapon Desc               object
Status                    object
Status Desc               object
Crm Cd 1                 float64
LOCATION                  object
LAT                      float64
LON                      float64
hour_occ                   int64
dtype: object

--- Valori nulli ---
  Mocodes: 375192 (12.18%)
  Vict Age: 632399 (20.54%)

### 7.1 Pulizia residua

La verifica finale ha rivelato 2 nulli in `Status` e 21 in `Crm Cd 1`, 
non emersi precedentemente. Data la quantità trascurabile (23 righe su 
3+ milioni), li rimuoviamo.

In [20]:
prima = len(df)
df = df.dropna(subset=['Status', 'Crm Cd 1'])
print(f"Righe rimosse: {prima - len(df)}")
print(f"Shape finale: {df.shape}")

Righe rimosse: 23
Shape finale: (3079424, 24)


## 8. Salvataggio del dataset pulito

Salviamo il DataFrame pulito come `crimes_clean.parquet` in `data/processed/`.
Questo file sarà il punto di partenza per tutti i notebook successivi 
(feature engineering, EDA, visualizzazioni, clustering).

In [21]:
df.to_parquet('../../data/processed/crimes_clean.parquet')
print("✅ Dataset pulito salvato in data/processed/crimes_clean.parquet")
print(f"   Righe: {len(df):,}")
print(f"   Colonne: {df.shape[1]}")

✅ Dataset pulito salvato in data/processed/crimes_clean.parquet
   Righe: 3,079,424
   Colonne: 24
